# Problem 1 — Qiskit notebook for a) b) c)

這份 notebook 針對 **QCAA HW1 Problem 1**，內容包含：

- **(a)** best configuration 的 training / test MSE vs epoch
- **(b)** 至少 4 組 hyperparameter configurations 比較表
- **(c)** target function 與 trained model output 的 2D Fourier spectrum 比較

你只需要依序執行各個 cell。若在 Colab 執行，先跑下一格安裝套件。

In [ ]:
# QCAA-HW1-VERIFIED
# If you are running on Colab or a fresh environment, uncomment the next line:
# !pip install -q qiskit qiskit-machine-learning torch pandas matplotlib

import sys
print(sys.version)


In [ ]:
# QCAA-HW1-VERIFIED
import math
import time
import copy
import itertools
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

print("Imports OK")


## 1. Data generation

下面直接沿用你給的 training / testing data 規則。

In [ ]:
# QCAA-HW1-VERIFIED
SEED = 10411304  # replace if needed

np.random.seed(SEED)
torch.manual_seed(SEED)

def target_function_torch(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(torch.exp(x[:, 0]) + x[:, 1])

def target_function_numpy(x1: np.ndarray, x2: np.ndarray) -> np.ndarray:
    return np.sin(np.exp(x1) + x2)

def generate_data(seed: int = SEED, n_train: int = 1000, n_test: int = 1000):
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_input = torch.zeros(n_train, 2, dtype=torch.float32)
    test_input = torch.zeros(n_test, 2, dtype=torch.float32)

    train_ranges = np.array([0.0, 0.5] * 2).reshape(2, 2)
    test_ranges = np.array([0.5, 1.0] * 2).reshape(2, 2)

    for i in range(2):
        train_input[:, i] = (
            torch.rand(n_train) * (train_ranges[i, 1] - train_ranges[i, 0]) + train_ranges[i, 0]
        )
        test_input[:, i] = (
            torch.rand(n_test) * (test_ranges[i, 1] - test_ranges[i, 0]) + test_ranges[i, 0]
        )

    train_label = target_function_torch(train_input)
    test_label = target_function_torch(test_input)
    return train_input, train_label, test_input, test_label

train_input, train_label, test_input, test_label = generate_data()
print("train_input:", train_input.shape)
print("test_input :", test_input.shape)
print("train_label range:", (float(train_label.min()), float(train_label.max())))
print("test_label  range:", (float(test_label.min()), float(test_label.max())))


## 2. Qiskit data-reuploading model

這裡用 **EstimatorQNN + TorchConnector** 做一個可訓練的 regression model。  
核心想法：

- 每一層都重新上傳輸入特徵 `x1, x2`
- 使用可切換的 encoding gate：`rx`, `ry`, `rz`
- 每層加入 trainable variational block
- 用平均 `Z` observable 輸出單一 scalar regression 值

In [ ]:
# QCAA-HW1-VERIFIED
@dataclass
class ExperimentConfig:
    name: str
    n_qubits: int
    n_layers: int
    encoding_gate: str  # "rx", "ry", "rz"
    lr: float = 0.03
    epochs: int = 120
    weight_scale: float = 0.1
    print_every: int = 20

def average_z_observable(n_qubits: int) -> SparsePauliOp:
    paulis = []
    coeffs = []
    for i in range(n_qubits):
        label = ["I"] * n_qubits
        label[n_qubits - 1 - i] = "Z"
        paulis.append("".join(label))
        coeffs.append(1.0 / n_qubits)
    return SparsePauliOp(paulis, coeffs=coeffs)

def apply_encoding_gate(qc: QuantumCircuit, gate_name: str, angle, qubit: int):
    gate_name = gate_name.lower()
    if gate_name == "rx":
        qc.rx(angle, qubit)
    elif gate_name == "ry":
        qc.ry(angle, qubit)
    elif gate_name == "rz":
        qc.rz(angle, qubit)
    else:
        raise ValueError(f"Unsupported encoding_gate={gate_name}")

def build_reuploading_circuit(n_qubits: int, n_layers: int, encoding_gate: str):
    x = ParameterVector("x", 2)
    w = ParameterVector("w", length=n_layers * n_qubits * 2)

    qc = QuantumCircuit(n_qubits)
    idx = 0

    for layer in range(n_layers):
        # Data reuploading block
        for q in range(n_qubits):
            feature = x[q % 2]  # alternate x0, x1, x0, x1, ...
            apply_encoding_gate(qc, encoding_gate, feature, q)

        # Trainable single-qubit block
        for q in range(n_qubits):
            qc.ry(w[idx], q)
            idx += 1
            qc.rz(w[idx], q)
            idx += 1

        # Ring entanglement
        if n_qubits > 1:
            for q in range(n_qubits - 1):
                qc.cx(q, q + 1)
            qc.cx(n_qubits - 1, 0)

    observable = average_z_observable(n_qubits)
    return qc, list(x), list(w), observable

def count_trainable_parameters_from_config(config: ExperimentConfig) -> int:
    return config.n_layers * config.n_qubits * 2

qc_example, x_params_example, w_params_example, obs_example = build_reuploading_circuit(
    n_qubits=2,
    n_layers=2,
    encoding_gate="rx",
)
print(qc_example.draw(output="text"))
print("num input params :", len(x_params_example))
print("num weight params:", len(w_params_example))
print("observable       :", obs_example)


In [ ]:
# QCAA-HW1-VERIFIED
class QiskitRegressor(nn.Module):
    def __init__(self, config: ExperimentConfig, seed: int = SEED):
        super().__init__()
        self.config = config
        self.circuit, self.input_params, self.weight_params, self.observable = build_reuploading_circuit(
            n_qubits=config.n_qubits,
            n_layers=config.n_layers,
            encoding_gate=config.encoding_gate,
        )

        estimator = StatevectorEstimator()
        qnn = EstimatorQNN(
            circuit=self.circuit,
            input_params=self.input_params,
            weight_params=self.weight_params,
            observables=self.observable,
            estimator=estimator,
            input_gradients=False,
        )

        rng = np.random.default_rng(seed)
        initial_weights = rng.normal(
            loc=0.0,
            scale=config.weight_scale,
            size=len(self.weight_params),
        )

        self.qnn_torch = TorchConnector(qnn, initial_weights=initial_weights)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.qnn_torch(x)
        if y.ndim == 2 and y.shape[1] == 1:
            y = y.squeeze(1)
        return y

    def count_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


## 3. Training / evaluation helpers

In [ ]:
# QCAA-HW1-VERIFIED
@torch.no_grad()
def evaluate_mse(model: nn.Module, x: torch.Tensor, y: torch.Tensor, criterion: nn.Module) -> float:
    model.eval()
    pred = model(x)
    loss = criterion(pred, y)
    return float(loss.item())

def train_one_config(config: ExperimentConfig, train_input, train_label, test_input, test_label, seed: int = SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = QiskitRegressor(config=config, seed=seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.MSELoss()

    history = {
        "epoch": [],
        "train_mse": [],
        "test_mse": [],
    }

    best_state = None
    best_test_mse = float("inf")
    best_epoch = -1

    start_time = time.perf_counter()

    for epoch in range(1, config.epochs + 1):
        model.train()
        optimizer.zero_grad()

        pred = model(train_input)
        loss = criterion(pred, train_label)
        loss.backward()
        optimizer.step()

        train_mse = evaluate_mse(model, train_input, train_label, criterion)
        test_mse = evaluate_mse(model, test_input, test_label, criterion)

        history["epoch"].append(epoch)
        history["train_mse"].append(train_mse)
        history["test_mse"].append(test_mse)

        if test_mse < best_test_mse:
            best_test_mse = test_mse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        if epoch == 1 or epoch % config.print_every == 0 or epoch == config.epochs:
            print(
                f"[{config.name}] epoch {epoch:3d}/{config.epochs} | "
                f"train MSE={train_mse:.6f} | test MSE={test_mse:.6f}"
            )

    elapsed = time.perf_counter() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    final_train_mse = evaluate_mse(model, train_input, train_label, criterion)
    final_test_mse = evaluate_mse(model, test_input, test_label, criterion)

    return {
        "config": config,
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_test_mse": best_test_mse,
        "final_train_mse": final_train_mse,
        "final_test_mse": final_test_mse,
        "n_trainable_params": model.count_trainable_parameters(),
        "elapsed_sec": elapsed,
    }

def build_default_configs():
    # 至少 4 組；你可以自行再加
    return [
        ExperimentConfig(name="cfg1_q2_L2_rx", n_qubits=2, n_layers=2, encoding_gate="rx", lr=0.03, epochs=120),
        ExperimentConfig(name="cfg2_q2_L4_rx", n_qubits=2, n_layers=4, encoding_gate="rx", lr=0.03, epochs=120),
        ExperimentConfig(name="cfg3_q4_L2_ry", n_qubits=4, n_layers=2, encoding_gate="ry", lr=0.02, epochs=120),
        ExperimentConfig(name="cfg4_q4_L4_ry", n_qubits=4, n_layers=4, encoding_gate="ry", lr=0.02, epochs=120),
    ]

def run_grid_search(configs, train_input, train_label, test_input, test_label, seed: int = SEED):
    results = []
    for i, config in enumerate(configs, start=1):
        print("=" * 90)
        print(f"Running {i}/{len(configs)}:", asdict(config))
        result = train_one_config(
            config=config,
            train_input=train_input,
            train_label=train_label,
            test_input=test_input,
            test_label=test_label,
            seed=seed,
        )
        results.append(result)
    results = sorted(results, key=lambda r: r["best_test_mse"])
    return results

def make_results_table(results):
    rows = []
    for r in results:
        cfg = r["config"]
        rows.append(
            {
                "name": cfg.name,
                "n_qubits": cfg.n_qubits,
                "n_layers": cfg.n_layers,
                "encoding_gate": cfg.encoding_gate,
                "lr": cfg.lr,
                "epochs": cfg.epochs,
                "train_MSE": r["final_train_mse"],
                "test_MSE": r["final_test_mse"],
                "best_test_MSE": r["best_test_mse"],
                "best_epoch": r["best_epoch"],
                "n_trainable_params": r["n_trainable_params"],
                "elapsed_sec": r["elapsed_sec"],
            }
        )
    return pd.DataFrame(rows).sort_values("best_test_MSE").reset_index(drop=True)

def get_best_result(results):
    return min(results, key=lambda r: r["best_test_mse"])


## 4. Run grid search for (b)

這一格會跑至少 4 組 configurations，並輸出 comparison table。

In [ ]:
# QCAA-HW1-VERIFIED
configs = build_default_configs()
results = run_grid_search(
    configs=configs,
    train_input=train_input,
    train_label=train_label,
    test_input=test_input,
    test_label=test_label,
    seed=SEED,
)

results_table = make_results_table(results)
results_table


## 5. Plot best model loss curves for (a)

In [ ]:
# QCAA-HW1-VERIFIED
best_result = get_best_result(results)
best_config = best_result["config"]

print("Best configuration:")
print(asdict(best_config))
print(f"Best epoch      : {best_result['best_epoch']}")
print(f"Final train MSE : {best_result['final_train_mse']:.6f}")
print(f"Final test MSE  : {best_result['final_test_mse']:.6f}")
print(f"Best test MSE   : {best_result['best_test_mse']:.6f}")
print(f"Trainable params: {best_result['n_trainable_params']}")
print(f"Elapsed seconds : {best_result['elapsed_sec']:.2f}")


In [ ]:
# QCAA-HW1-VERIFIED
def plot_loss_curves(result):
    history = result["history"]
    cfg = result["config"]

    plt.figure(figsize=(8, 5))
    plt.plot(history["epoch"], history["train_mse"], label="Train MSE")
    plt.plot(history["epoch"], history["test_mse"], label="Test MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(f"Problem 1 Loss Curves — {cfg.name}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_loss_curves(best_result)


## 6. Fourier spectrum for (c)

題目附錄建議在 test domain 上建立 regular grid，然後做 2D FFT。  
下面會同時比較：

- target function `sin(exp(x1)+x2)`
- best trained model output

In [ ]:
# QCAA-HW1-VERIFIED
@torch.no_grad()
def evaluate_model_on_grid(model, x1_min=0.5, x1_max=1.0, x2_min=0.5, x2_max=1.0, grid_size=64):
    model.eval()

    x1 = np.linspace(x1_min, x1_max, grid_size)
    x2 = np.linspace(x2_min, x2_max, grid_size)
    X1, X2 = np.meshgrid(x1, x2)

    grid_points = np.stack([X1.ravel(), X2.ravel()], axis=1)
    grid_tensor = torch.tensor(grid_points, dtype=torch.float32)

    pred = model(grid_tensor).detach().cpu().numpy().reshape(grid_size, grid_size)
    return X1, X2, pred

def compute_2d_fourier_spectrum(values: np.ndarray) -> np.ndarray:
    spectrum = np.fft.fft2(values)
    magnitude = np.abs(np.fft.fftshift(spectrum))
    return magnitude

def top_frequency_peaks(magnitude: np.ndarray, k: int = 8):
    mag = magnitude.copy()
    center = (mag.shape[0] // 2, mag.shape[1] // 2)
    mag[center] = 0.0  # ignore DC peak

    flat_idx = np.argpartition(mag.ravel(), -k)[-k:]
    flat_idx = flat_idx[np.argsort(mag.ravel()[flat_idx])[::-1]]

    peaks = []
    for idx in flat_idx:
        i, j = np.unravel_index(idx, mag.shape)
        peaks.append({"index": (int(i), int(j)), "magnitude": float(mag[i, j])})
    return peaks

def plot_fourier_comparison(best_result, grid_size: int = 64):
    model = best_result["model"]

    x1 = np.linspace(0.5, 1.0, grid_size)
    x2 = np.linspace(0.5, 1.0, grid_size)
    X1, X2 = np.meshgrid(x1, x2)

    target_values = target_function_numpy(X1, X2)
    _, _, pred_values = evaluate_model_on_grid(model, grid_size=grid_size)

    target_spectrum = compute_2d_fourier_spectrum(target_values)
    pred_spectrum = compute_2d_fourier_spectrum(pred_values)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    im0 = axes[0, 0].imshow(target_values, origin="lower", aspect="auto", extent=[0.5, 1.0, 0.5, 1.0])
    axes[0, 0].set_title("Target function on test domain")
    axes[0, 0].set_xlabel("x1")
    axes[0, 0].set_ylabel("x2")
    plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

    im1 = axes[0, 1].imshow(pred_values, origin="lower", aspect="auto", extent=[0.5, 1.0, 0.5, 1.0])
    axes[0, 1].set_title("Best model output on test domain")
    axes[0, 1].set_xlabel("x1")
    axes[0, 1].set_ylabel("x2")
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    im2 = axes[1, 0].imshow(np.log1p(target_spectrum), origin="lower", aspect="auto")
    axes[1, 0].set_title("Target Fourier spectrum (log scale)")
    axes[1, 0].set_xlabel("frequency index")
    axes[1, 0].set_ylabel("frequency index")
    plt.colorbar(im2, ax=axes[1, 0], fraction=0.046)

    im3 = axes[1, 1].imshow(np.log1p(pred_spectrum), origin="lower", aspect="auto")
    axes[1, 1].set_title("Model Fourier spectrum (log scale)")
    axes[1, 1].set_xlabel("frequency index")
    axes[1, 1].set_ylabel("frequency index")
    plt.colorbar(im3, ax=axes[1, 1], fraction=0.046)

    plt.tight_layout()
    plt.show()

    return {
        "target_values": target_values,
        "pred_values": pred_values,
        "target_spectrum": target_spectrum,
        "pred_spectrum": pred_spectrum,
        "target_top_peaks": top_frequency_peaks(target_spectrum, k=8),
        "pred_top_peaks": top_frequency_peaks(pred_spectrum, k=8),
    }

fourier_outputs = plot_fourier_comparison(best_result, grid_size=64)


In [ ]:
# QCAA-HW1-VERIFIED
print("Top target spectrum peaks:")
pd.DataFrame(fourier_outputs["target_top_peaks"])

print("\nTop model spectrum peaks:")
pd.DataFrame(fourier_outputs["pred_top_peaks"])


## 7. Optional: export result table to CSV

In [ ]:
# QCAA-HW1-VERIFIED
results_table.to_csv("problem1_qiskit_results.csv", index=False)
print("Saved: problem1_qiskit_results.csv")


## 8. Notes for your report

你可以直接根據 notebook 輸出來寫：

- **(a)** 貼 best config 的 MSE vs epoch 圖
- **(b)** 貼 `results_table`
- **(c)** 貼 Fourier comparison 圖，並討論：
  - model 是否抓到 target 的主要頻率峰值
  - 較深的 reuploading layers 是否更能捕捉較高頻率成分
  - 如果 spectrum 少掉某些 peaks，通常代表 model expressivity 還不夠